# Strategy Validation Notebook

Interactively validate the pure `src/core` interfaces used by the alpha pipeline.

**Workflow:** load model/features/prices → generate scores → inspect TopK/BottomK → build rolling portfolio → compute spread metrics.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from src.core.signals import generate_scores, generate_scores_panel
from src.core.selection import select_topk, select_bottomk
from src.core.portfolio import build_rolling_portfolio
from src.core.metrics import compute_spread

## 1. Load model, features, and close prices


In [ ]:
# --- EDIT THESE PATHS ---
MODEL_PATH = ROOT / 'models' / 'lgbm_latest.pkl'
FEATURE_PATH = ROOT / 'data' / 'features_test.parquet'
PRICE_PATH = ROOT / 'data' / 'close_prices.parquet'
BENCH_TICKER = 'SPY'
# -------------------------

model = joblib.load(MODEL_PATH)
feature_panel = pd.read_parquet(FEATURE_PATH)  # MultiIndex: (datetime, instrument)
close = pd.read_parquet(PRICE_PATH)            # Wide frame: datetime x instrument

print(f'Feature panel: {feature_panel.shape}')
print(f'Close prices : {close.shape}')

## 2. Prepare return panel for `build_rolling_portfolio`


In [ ]:
return_panel = close.pct_change().stack().rename('return').to_frame()
return_panel.index.names = ['datetime', 'instrument']

benchmark_returns = None
if BENCH_TICKER in close.columns:
    benchmark_returns = close[BENCH_TICKER].pct_change().rename('benchmark_return')
    return_panel = return_panel.drop(index=BENCH_TICKER, level='instrument', errors='ignore')

return_panel.head()

## 3. Generate and inspect scores for a single date


In [ ]:
SAMPLE_DATE = feature_panel.index.get_level_values(0).unique()[-1]
feature_slice = feature_panel.xs(SAMPLE_DATE, level=0)
scores = generate_scores(model, feature_slice)

print(f'Sample date: {SAMPLE_DATE}')
print(scores.describe())
scores.hist(bins=50)
plt.title(f'Score distribution — {SAMPLE_DATE}')
plt.xlabel('score')
plt.ylabel('count')
plt.tight_layout()
plt.show()

## 4. Inspect TopK / BottomK selection


In [ ]:
K = 10
top_tickers = select_topk(scores, k=K)
bottom_tickers = select_bottomk(scores, k=K)

print(f'Long  ({len(top_tickers)}): {top_tickers}')
print(f'Short ({len(bottom_tickers)}): {bottom_tickers}')

pd.DataFrame({
    'long_score': scores.reindex(top_tickers).reset_index(drop=True),
    'short_score': scores.reindex(bottom_tickers).reset_index(drop=True),
})

## 5. Build rolling portfolio and compute spread metrics


In [ ]:
score_panel = generate_scores_panel(model, feature_panel)

portfolio = build_rolling_portfolio(
    score_panel=score_panel,
    return_panel=return_panel,
    k=K,
    holding_days=5,
    benchmark_returns=benchmark_returns,
)

metrics = compute_spread(
    long_returns=portfolio.long_returns,
    short_returns=portfolio.short_returns,
    bench_returns=portfolio.bench_returns if len(portfolio.bench_returns) else None,
)

for name, value in metrics.items():
    if not isinstance(value, pd.Series):
        print(f'{name:<20}: {value:.4f}')

metrics['spread_equity'].plot(title='Long-short spread equity', figsize=(10, 4))
plt.axhline(1.0, color='grey', linestyle='--')
plt.tight_layout()
plt.show()

## 6. Parameter sweep


In [ ]:
rows = []
for k_val in [5, 10, 20, 30]:
    for hold in [3, 5, 10]:
        pf = build_rolling_portfolio(
            score_panel=score_panel,
            return_panel=return_panel,
            k=k_val,
            holding_days=hold,
            benchmark_returns=benchmark_returns,
        )
        m = compute_spread(
            pf.long_returns,
            pf.short_returns,
            pf.bench_returns if len(pf.bench_returns) else None,
        )
        rows.append({
            'k': k_val,
            'holding_days': hold,
            'spread_mean': m['spread_mean'],
            'spread_sharpe': m['spread_sharpe'],
            'alpha_long': m['alpha_long'],
            'alpha_short': m['alpha_short'],
        })

pd.DataFrame(rows).sort_values('spread_sharpe', ascending=False)